<a href="https://colab.research.google.com/github/JordanDCunha/Introduction-to-Machine-Learning-with-Python/blob/main/Chapter4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 4. Representing Data and Engineering Features

## Key Ideas
- Machine learning data is often **not** just continuous numerical features.
- Many real-world datasets include **categorical (discrete) features**.
- Categorical features:
  - Are usually **non-numeric**
  - Have **no natural ordering**
  - Do not vary continuously

## Continuous vs Categorical Features
- **Continuous features**:
  - Examples: pixel brightness, flower size, measurements
  - Can take on any value within a range
- **Categorical features**:
  - Examples: product brand, color, department (books, clothing, hardware)
  - Belong to one category only
  - No concept of “in between” values

## Importance of Data Representation
- How features are represented can **greatly affect model performance**
- Scaling matters:
  - Representing data in inches vs centimeters can change results
  - Proper scaling (e.g., unit variance) improves many models
- Feature augmentation can help:
  - Feature interactions
  - Polynomial features

## Feature Engineering
- **Feature engineering** = choosing the best way to represent data
- Often has **more impact than model hyperparameters**
- A core responsibility of data scientists and ML practitioners

## What This Chapter Covers
- Handling and representing **categorical features**
- Useful feature transformations for specific models
- How feature representation influences learning outcomes


## 4.1 Categorical Variables

### Problem Setup (Adult Income Dataset)
- Dataset derived from the 1994 US census
- Goal: predict whether income is **<= $50K** or **> $50K**
- This is a **binary classification** task
- Regression is possible but harder and less interpretable for this task

### Feature Types in the Dataset
- **Continuous features**:
  - `age`
  - `hours-per-week`
- **Categorical features**:
  - `workclass`
  - `education`
  - `gender`
  - `occupation`
- Categorical features:
  - Come from a fixed set of values
  - Represent qualitative properties
  - Cannot be used directly in linear models

### Why Encoding Is Necessary
- Logistic regression computes a weighted sum of **numerical** inputs
- Strings like `"Bachelors"` or `"Masters"` cannot be used directly
- Categorical features must be converted into numeric form


## 4.1.1 One-Hot Encoding (Dummy Variables)

### Core Idea
- Replace a categorical feature with multiple **binary (0/1) features**
- Each category gets its own feature
- Exactly **one feature is 1** per data point (one-out-of-N)

### Example: Workclass
- Categories:
  - Government Employee
  - Private Employee
  - Self Employed
  - Self Employed Incorporated
- Result:
  - Four binary features
  - Original categorical column is dropped

### Why One-Hot Encoding Works
- Binary values fit naturally into linear models
- No artificial ordering is introduced
- Allows models to learn separate weights per category

### Note on Statistics vs ML
- Statistics often uses **k–1** dummy variables
- Here we use **k variables** for simplicity


In [ ]:
import os
# The file has no headers naming the columns, so we pass header=None
# and provide the column names explicitly in "names"
adult_path = os.path.join(mglearn.datasets.DATA_PATH, "adult.data")
data = pd.read_csv(
    adult_path, header=None, index_col=False,
    names=['age', 'workclass', 'fnlwgt', 'education',  'education-num',
           'marital-status', 'occupation', 'relationship', 'race', 'gender',
           'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
           'income'])

# Select a subset of columns
data = data[['age', 'workclass', 'education', 'gender',
             'hours-per-week', 'occupation', 'income']]

display(data.head())


### Checking Categorical Data Quality
- Human-entered data may contain:
  - Spelling inconsistencies
  - Capitalization differences
  - Unexpected categories
- Always inspect unique values before encoding
- `value_counts()` helps identify issues


In [ ]:
print(data.gender.value_counts())


### Using pandas.get_dummies
- Automatically one-hot encodes:
  - String columns
  - Categorical columns
- Leaves continuous numeric features unchanged
- Produces a fully numeric dataset usable by scikit-learn


In [ ]:
print("Original features:\n", list(data.columns), "\n")
data_dummies = pd.get_dummies(data)
print("Features after get_dummies:\n", list(data_dummies.columns))


In [ ]:
data_dummies.head()


### Separating Features and Target
- Target variable (`income`) is now encoded as:
  - `income_ <=50K`
  - `income_ >50K`
- Must remove target columns from feature matrix
- Including target info in features causes **data leakage**


In [ ]:
features = data_dummies.loc[:, 'age':'occupation_ Transport-moving']

X = features.values
y = data_dummies['income_ >50K'].values

print("X.shape: {}  y.shape: {}".format(X.shape, y.shape))


### Training a Model
- Data is now fully numeric
- Can be used directly with scikit-learn models
- Logistic regression used as baseline classifier


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

logreg = LogisticRegression()
logreg.fit(X_train, y_train)

print("Test score: {:.2f}".format(logreg.score(X_test, y_test)))


### Important Warning: Train/Test Consistency
- One-hot encoding must be **consistent**
- Training and test sets must have:
  - Same columns
  - Same column meanings
- Best practices:
  - Apply `get_dummies` before splitting
  - Or ensure columns are aligned manually


## 4.1.2 Numbers Can Encode Categoricals

### Integers ≠ Continuous Features
- Categorical features are often stored as integers
- Numeric values do NOT imply ordering or continuity
- Example:
  - Workclass encoded as 0, 1, 2, 3
  - These represent categories, not magnitudes

### Key Rule
- If no meaningful ordering exists:
  - Treat the feature as categorical
  - Use one-hot encoding


In [ ]:
demo_df = pd.DataFrame({
    'Integer Feature': [0, 1, 2, 1],
    'Categorical Feature': ['socks', 'fox', 'socks', 'box']
})

display(demo_df)


In [ ]:
display(pd.get_dummies(demo_df))


### Encoding Integer Categoricals Explicitly
- pandas treats integers as continuous by default
- To one-hot encode integers:
  - Convert them to strings
  - Or specify columns explicitly


In [ ]:
demo_df['Integer Feature'] = demo_df['Integer Feature'].astype(str)

display(pd.get_dummies(
    demo_df,
    columns=['Integer Feature', 'Categorical Feature']
))


# 4.2 OneHotEncoder and ColumnTransformer  
## Categorical Variables with scikit-learn


### Why use scikit-learn for categorical variables?

- scikit-learn provides **OneHotEncoder** for one-hot encoding
- Advantage over pandas:
  - Ensures **consistent feature encoding** between training and test sets
- OneHotEncoder:
  - Applies encoding to **all input columns**
  - Returns a **NumPy array**, not a DataFrame
- In practice, datasets often contain:
  - **Categorical features**
  - **Continuous features**
- To handle mixed data types, scikit-learn provides **ColumnTransformer**


In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd


In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd


### Applying OneHotEncoder

- OneHotEncoder treats **all columns as categorical**
- Both integer and string features are encoded
- Output is a NumPy array without column names


In [ ]:
ohe = OneHotEncoder(sparse=False)
encoded = ohe.fit_transform(demo_df)
encoded


### Understanding the encoded output

- Each unique category becomes a **binary (0/1) feature**
- Columns are ordered by feature index, then category
- No column labels are shown by default


In [ ]:
ohe.get_feature_names()


### Interpreting feature names

- `x0_*` → categories from the **first column**
- `x1_*` → categories from the **second column**
- Example:
  - `x0_0`, `x0_1`, `x0_2` represent integer categories
  - `x1_box`, `x1_fox`, `x1_socks` represent string categories


### Why ColumnTransformer is needed

- Real datasets usually mix:
  - **Continuous features** (e.g., age)
  - **Categorical features** (e.g., workclass)
- OneHotEncoder alone assumes *all* features are categorical
- ColumnTransformer allows:
  - Different preprocessing per column
  - Concatenation of transformed features
- Prevents preprocessing errors and feature mismatch


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler


### Adult Census dataset example

- Target: predict whether income is `>50K`
- Continuous features:
  - `age`
  - `hours-per-week`
- Categorical features:
  - `workclass`
  - `education`
  - `gender`
  - `occupation`
- Strategy:
  - Scale continuous features
  - One-hot encode categorical features


In [ ]:
# example: data already loaded earlier as a pandas DataFrame called `data`
data.head()


In [ ]:
ct = ColumnTransformer(
    transformers=[
        ("scaling", StandardScaler(), ['age', 'hours-per-week']),
        ("onehot", OneHotEncoder(sparse=False),
         ['workclass', 'education', 'gender', 'occupation'])
    ]
)


### Train-test split using DataFrames

- ColumnTransformer relies on **column names**
- Keep features as a DataFrame (not NumPy arrays)
- Separate the target variable before training


In [ ]:
from sklearn.model_selection import train_test_split

# separate features and target
data_features = data.drop("income", axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    data_features, data.income, random_state=0
)


### Applying the ColumnTransformer

- `fit` learns scaling parameters and categories
- `transform` applies preprocessing
- Output is a NumPy array ready for modeling


In [ ]:
ct.fit(X_train)

X_train_trans = ct.transform(X_train)
X_train_trans.shape


### Resulting feature space

- Total features: **44**
- Matches pandas `get_dummies`
- Continuous features are scaled
- Categorical features are one-hot encoded


### Training a Logistic Regression model


In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression()
logreg.fit(X_train_trans, y_train)

X_test_trans = ct.transform(X_test)
logreg.score(X_test_trans, y_test)


### Model performance

- Accuracy ≈ **0.81**
- Scaling did not improve accuracy here
- Main benefit is:
  - Clean preprocessing
  - Reproducibility
  - Safe train/test handling


### Accessing transformers inside ColumnTransformer

- Individual transformers are stored in:
  - `named_transformers_`
- Useful for inspection and debugging


In [ ]:
ct.named_transformers_.onehot


### Key takeaways

- **OneHotEncoder**
  - Encodes categorical features into binary vectors
  - Works with strings and integers
- **ColumnTransformer**
  - Essential for mixed feature types
  - Prevents semantic feature mismatches
- Recommended workflow:
  - DataFrame → ColumnTransformer → Model


# 4.3 Convenient ColumnTransformer Creation  
## Using `make_columntransformer`


### Why use `make_columntransformer`?

- Creating a `ColumnTransformer` can be verbose
- Often we **don’t care about custom step names**
- `make_columntransformer`:
  - Automatically creates a `ColumnTransformer`
  - Automatically names each step based on the transformer class
  - Produces cleaner and more readable code
- Functionally equivalent to manually defining `ColumnTransformer`


In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


### Syntax of `make_columntransformer`

- Each transformation is given as a **tuple**:
  - `(columns, transformer)`
- No need to provide explicit step names
- Order matters: transformations are applied and concatenated in order


In [ ]:
ct = make_column_transformer(
    (['age', 'hours-per-week'], StandardScaler()),
    (['workclass', 'education', 'gender', 'occupation'],
     OneHotEncoder(sparse=False))
)


### What this ColumnTransformer does

- **StandardScaler**
  - Applied to:
    - `age`
    - `hours-per-week`
- **OneHotEncoder**
  - Applied to:
    - `workclass`
    - `education`
    - `gender`
    - `occupation`
- Output:
  - Scaled continuous features
  - One-hot encoded categorical features
  - Concatenated into a single NumPy array


### Using the transformer in a workflow

- Same API as other scikit-learn transformers:
  - `fit(X_train)`
  - `transform(X_train)`
  - `transform(X_test)`
- Works directly with pandas DataFrames
- Preserves correct preprocessing for train/test splits


In [ ]:
# assuming data_features and y already exist
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data_features, data.income, random_state=0
)

ct.fit(X_train)

X_train_trans = ct.transform(X_train)
X_test_trans = ct.transform(X_test)

X_train_trans.shape


### Advantages of `make_columntransformer`

- Less boilerplate code
- Cleaner, more readable notebooks
- Automatic naming of steps
- Ideal for:
  - Prototyping
  - Teaching
  - Most real-world pipelines


### Disadvantage (important exam / interview point)

- In scikit-learn version 0.20:
  - It is **not easy to trace**
    - Which input columns map to which output columns
- This can make:
  - Debugging harder
  - Feature interpretation more difficult
- Especially relevant when:
  - Inspecting coefficients of linear models


### Practical takeaway

- Use `make_columntransformer` when:
  - You want concise code
  - You don’t need custom transformer names
- Use `ColumnTransformer` when:
  - You need explicit control
  - You want clearer feature-to-column mapping


# 4.4 Binning, Discretization, Linear Models, and Trees


### Key idea

- The **best feature representation** depends on:
  - The **data**
  - The **model** being used
- Different model families behave very differently:
  - **Linear models**
  - **Tree-based models** (decision trees, random forests, gradient boosting)
- Feature engineering that helps one model may **hurt or not help** another


### Linear models vs tree-based models

- **Linear models**
  - Can only model **linear relationships**
  - With one feature → straight line
- **Decision trees**
  - Can model **highly nonlinear relationships**
  - Automatically split the feature space
- Model expressiveness depends heavily on **feature representation**


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
import numpy as np
import matplotlib.pyplot as plt
import mglearn


In [ ]:
# create the wave dataset
X, y = mglearn.datasets.make_wave(n_samples=120)
line = np.linspace(-3, 3, 1000, endpoint=False).reshape(-1, 1)

# decision tree
reg = DecisionTreeRegressor(min_samples_leaf=3).fit(X, y)
plt.plot(line, reg.predict(line), label="decision tree")

# linear regression
reg = LinearRegression().fit(X, y)
plt.plot(line, reg.predict(line), label="linear regression")

plt.plot(X[:, 0], y, 'o', c='k')
plt.ylabel("Regression output")
plt.xlabel("Input feature")
plt.legend(loc="best")


### Observation

- Linear regression:
  - Fits a **straight line**
  - Limited expressiveness
- Decision tree:
  - Fits a **piecewise constant nonlinear function**
  - Much more flexible
- But flexibility depends on **how the data is represented**


### Binning (Discretization)

- **Binning** (or discretization):
  - Converts a continuous feature into **discrete intervals**
- Each data point is represented by:
  - Which **bin** it falls into
- Common strategies:
  - **Uniform width bins**
  - **Quantile-based bins**
- Implemented in scikit-learn via `KBinsDiscretizer`


from sklearn.preprocessing import KBinsDiscretizer


In [ ]:
# create 10 uniform-width bins
kb = KBinsDiscretizer(n_bins=10, strategy='uniform')
kb.fit(X)

print("Bin edges:\n", kb.bin_edges_)


### What the bins represent

- The input range (≈ −3 to 3) is split into **10 equal-width intervals**
- Each bin contains values within a specific range
- `bin_edges_`:
  - Stores the boundaries of each bin
  - One list per feature


### Encoding data points by bins

- Each data point is encoded by:
  - Which bin it belongs to
- Default behavior:
  - **One-hot encoding**
  - Produces a **sparse matrix**
- With 10 bins → 10-dimensional feature space


In [ ]:
X_binned = kb.transform(X)
X_binned


### Inspecting the encoding

- Exactly **one feature is 1** per data point
- All other bin features are 0
- This turns a continuous feature into a **categorical-like representation**


In [ ]:
print(X[:10])
X_binned.toarray()[:10]


### Simplifying output for demonstration

- Use dense one-hot encoding:
  - Easier to print and inspect
- Set:
  - `encode='onehot-dense'`


In [ ]:
kb = KBinsDiscretizer(
    n_bins=10,
    strategy='uniform',
    encode='onehot-dense'
)

kb.fit(X)
X_binned = kb.transform(X)


### Training models on binned data

- Train:
  - Linear regression
  - Decision tree
- Both models now see:
  - One-hot encoded bin features


In [ ]:
line_binned = kb.transform(line)

# linear regression on binned data
reg = LinearRegression().fit(X_binned, y)
plt.plot(line, reg.predict(line_binned), label="linear regression binned")

# decision tree on binned data
reg = DecisionTreeRegressor(min_samples_split=3).fit(X_binned, y)
plt.plot(line, reg.predict(line_binned), label="decision tree binned")

plt.plot(X[:, 0], y, 'o', c='k')
plt.vlines(kb.bin_edges_[0], -3, 3, linewidth=1, alpha=.2)
plt.legend(loc="best")
plt.ylabel("Regression output")
plt.xlabel("Input feature")


### Key result

- **Linear regression and decision tree make identical predictions**
- Both predict:
  - A **constant value per bin**
- Why?
  - Features are constant within each bin
  - Models cannot distinguish points inside the same bin


### Effect of binning on models

- **Linear model**
  - Becomes **much more flexible**
  - Can now model nonlinear relationships
- **Decision tree**
  - Becomes **less flexible**
  - Already knows how to split optimally
- Trees effectively learn their own binning automatically


### When binning is useful

- Binning is helpful when:
  - You want to use a **linear model**
  - Data is **large or high-dimensional**
  - Some features have **nonlinear effects**
- Binning is usually **not helpful** for:
  - Decision trees
  - Random forests
  - Gradient boosted trees


### Takeaway

- Feature engineering must consider:
  - **Model assumptions**
  - **Model flexibility**
- Same transformation can:
  - Greatly help one model
  - Hurt or not affect another
- Binning is a powerful tool to:
  - Increase expressiveness of linear models


### 4.5 Interactions and Polynomials

- Feature engineering can enrich data representations, especially for **linear models**.
- Two common techniques:
  - **Interaction features** (products of features)
  - **Polynomial features** (powers of features)
- These methods allow linear models to capture **nonlinear relationships**.
- Commonly used in statistics and practical machine learning applications.


- A linear model trained on **binned data** learns only constant offsets per bin.
- Linear models can also learn **slopes**, but binned data alone prevents this.
- Adding the **original feature** back enables a global slope across bins.
- This creates a combined dataset with:
  - Bin indicators
  - Original continuous feature


In [ ]:
X_combined = np.hstack([X, X_binned])
print(X_combined.shape)


In [ ]:
reg = LinearRegression().fit(X_combined, y)

line_combined = np.hstack([line, line_binned])
plt.plot(line, reg.predict(line_combined), label='linear regression combined')

plt.vlines(kb.bin_edges_[0], -3, 3, linewidth=1, alpha=.2)
plt.legend(loc="best")
plt.ylabel("Regression output")
plt.xlabel("Input feature")
plt.plot(X[:, 0], y, 'o', c='k')


- The model learns:
  - One offset per bin
  - A **single shared slope**
- The shared slope is often not expressive enough.
- Ideally, each bin should have its **own slope**.


- Interaction features are created by multiplying:
  - Bin indicator × original feature
- This results in:
  - A separate copy of the original feature **per bin**
  - Zero values outside the bin
- Enables **independent slopes for each bin**.


In [ ]:
X_product = np.hstack([X_binned, X * X_binned])
print(X_product.shape)


In [ ]:
reg = LinearRegression().fit(X_product, y)

line_product = np.hstack([line_binned, line * line_binned])
plt.plot(line, reg.predict(line_product), label='linear regression product')

plt.vlines(kb.bin_edges_[0], -3, 3, linewidth=1, alpha=.2)
plt.plot(X[:, 0], y, 'o', c='k')
plt.ylabel("Regression output")
plt.xlabel("Input feature")
plt.legend(loc="best")


- Another way to expand continuous features is using **polynomials**.
- For a feature x, common polynomial terms include:
  - x², x³, x⁴, ...
- Implemented in scikit-learn via `PolynomialFeatures`.
- Polynomial regression = linear regression on polynomial features.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=10, include_bias=False)
poly.fit(X)
X_poly = poly.transform(X)


In [ ]:
print("X_poly.shape:", X_poly.shape)


- The first column of `X_poly` equals the original feature.
- Remaining columns are increasing powers of that feature.
- High-degree polynomials can produce:
  - Very large values
  - Numerical instability


In [ ]:
print("Entries of X:\n", X[:5])
print("Entries of X_poly:\n", X_poly[:5])


In [ ]:
print("Polynomial feature names:\n", poly.get_feature_names())


- Using polynomial features with linear regression yields:
  - Smooth nonlinear fits
- High-degree polynomials:
  - Can overfit
  - Behave unpredictably near data boundaries


In [ ]:
reg = LinearRegression().fit(X_poly, y)

line_poly = poly.transform(line)
plt.plot(line, reg.predict(line_poly), label='polynomial linear regression')
plt.plot(X[:, 0], y, 'o', c='k')
plt.ylabel("Regression output")
plt.xlabel("Input feature")
plt.legend(loc="best")


- Kernel SVMs can model complex nonlinear patterns
- Do not require explicit feature transformations
- RBF kernel achieves similar flexibility to polynomial regression


In [ ]:
from sklearn.svm import SVR

for gamma in [1, 10]:
    svr = SVR(gamma=gamma).fit(X, y)
    plt.plot(line, svr.predict(line), label=f'SVR gamma={gamma}')

plt.plot(X[:, 0], y, 'o', c='k')
plt.ylabel("Regression output")
plt.xlabel("Input feature")
plt.legend(loc="best")


- Polynomial and interaction features applied to real-world data
- Dataset:
  - 13 original features
- Data is scaled using MinMaxScaler
- Polynomial features expanded up to degree 2


In [ ]:
from sklearn.datasets import load_boston
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

boston = load_boston()
X_train, X_test, y_train, y_test = train_test_split(
    boston.data, boston.target, random_state=0)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
poly = PolynomialFeatures(degree=2).fit(X_train_scaled)
X_train_poly = poly.transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

print("X_train.shape:", X_train.shape)
print("X_train_poly.shape:", X_train_poly.shape)


- Polynomial expansion increased features from 13 → 105
- Linear models benefit significantly from interactions
- Tree-based models already capture interactions naturally


In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge().fit(X_train_scaled, y_train)
print("Score without interactions:", ridge.score(X_test_scaled, y_test))

ridge = Ridge().fit(X_train_poly, y_train)
print("Score with interactions:", ridge.score(X_test_poly, y_test))


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100).fit(X_train_scaled, y_train)
print("Score without interactions:", rf.score(X_test_scaled, y_test))

rf = RandomForestRegressor(n_estimators=100).fit(X_train_poly, y_train)
print("Score with interactions:", rf.score(X_test_poly, y_test))


- Interaction and polynomial features:
  - Greatly help **linear models**
  - Can hurt **tree-based models**
- Feature engineering impact can exceed model choice
- Always consider:
  - Model complexity
  - Feature explosion
  - Overfitting risk


### 4.6 Univariate Nonlinear Transformations

- Polynomial features (x², x³, …) help linear models capture nonlinear patterns.
- Other useful transformations include:
  - `log`
  - `exp`
  - `sin` / `cos`
- These are especially important for:
  - Linear models
  - Neural networks
- Tree-based models mostly care about **feature ordering**, not scale or distribution.


- Linear models struggle when:
  - Features have nonlinear relationships with the target
  - Feature distributions are highly skewed
- Transformations like `log` and `exp`:
  - Compress large values
  - Reduce skew
  - Make relationships easier to model
- `sin` and `cos` are useful for:
  - Periodic or cyclical data (time, seasons, angles)


- Most models perform best when features are roughly Gaussian ("bell-shaped").
- Many real-world features are:
  - Highly skewed
  - Long-tailed
  - Non-negative
- Applying transformations is a simple and effective way to:
  - Stabilize variance
  - Reduce extreme outliers


- Count data examples:
  - Number of logins
  - Number of clicks
  - Number of purchases
- Properties of count data:
  - Integer-valued
  - Non-negative
  - Often highly skewed
- Linear models usually struggle with raw count features.


In [ ]:
rnd = np.random.RandomState(0)
X_org = rnd.normal(size=(1000, 3))
w = rnd.normal(size=3)

X = rnd.poisson(10 * np.exp(X_org))
y = np.dot(X_org, w)


- Feature values are:
  - Positive
  - Integer-valued
- Distribution shape is not obvious from raw samples alone.
- Counting occurrences reveals the underlying structure.


In [ ]:
print("Number of feature appearances:\n{}".format(np.bincount(X[:, 0])))


- Most values are small and frequent.
- Larger values appear rarely but exist as outliers.
- This "long-tail" distribution is very common in practice.
- Such distributions are difficult for linear models to handle.


In [ ]:
bins = np.bincount(X[:, 0])
plt.bar(range(len(bins)), bins, color='grey')
plt.ylabel("Number of appearances")
plt.xlabel("Value")


- Ridge regression is applied to raw count features.
- Performance is limited due to:
  - Skewed distributions
  - Extreme outliers


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

score = Ridge().fit(X_train, y_train).score(X_test, y_test)
print("Test score: {:.3f}".format(score))


- Log transformation reduces skew and compresses large values.
- Since log(0) is undefined:
  - Use `log(X + 1)`
- This transformation often improves linear model performance.


In [ ]:
X_train_log = np.log(X_train + 1)
X_test_log = np.log(X_test + 1)


- Distribution becomes:
  - More symmetric
  - Less affected by outliers
- Feature values are now easier for linear models to learn from.


In [ ]:
plt.hist(X_train_log[:, 0], bins=25, color='gray')
plt.ylabel("Number of appearances")
plt.xlabel("Value")


- Ridge regression is retrained on log-transformed features.
- Performance improves significantly.


In [ ]:
score = Ridge().fit(X_train_log, y_train).score(X_test_log, y_test)
print("Test score: {:.3f}".format(score))


- Choosing transformations is partly an art.
- In real datasets:
  - Only some features may need transformation
  - Different features may need different transformations
- Transformations are:
  - Crucial for linear models
  - Often unnecessary for tree-based models


- Sometimes transforming the target y helps in regression.
- Common example:
  - Predicting counts (orders, visits, clicks)
- Using `log(y + 1)` often improves results.


- Feature transformations can drastically affect model performance.
- Especially impactful for:
  - Linear models
  - Naive Bayes
- Tree-based models usually:
  - Learn interactions automatically
  - Require fewer explicit transformations
- Other models (SVMs, kNN, neural networks):
  - May benefit, but effects are less predictable


### 4.7 Automatic Feature Selection

- Creating many new features increases model complexity.
- More features → higher risk of overfitting.
- Feature selection:
  - Keeps only the most useful features
  - Produces simpler models
  - Can improve generalization
- All feature selection methods discussed here are **supervised**:
  - They require target labels
  - Must be fit only on training data


There are three main strategies for automatic feature selection:

1. **Univariate statistics**
2. **Model-based selection**
3. **Iterative selection**

Each method has different trade-offs in speed, complexity, and effectiveness.


### 4.7.1 Univariate Statistics

- Measure statistical relationship between:
  - Each individual feature
  - The target variable
- Classification case:
  - Known as ANOVA
- Regression case:
  - Uses correlation-based tests


- Each feature is evaluated **independently**
- Pros:
  - Very fast
  - No model required
- Cons:
  - Cannot capture feature interactions
  - May discard features useful only in combination
- Independent of the final model used


- Common statistical tests:
  - `f_classif` (classification, default)
  - `f_regression` (regression)
- Feature selection methods:
  - `SelectKBest`: keeps top k features
  - `SelectPercentile`: keeps top percentage


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectPercentile
from sklearn.model_selection import train_test_split
import numpy as np

cancer = load_breast_cancer()

rng = np.random.RandomState(42)
noise = rng.normal(size=(len(cancer.data), 50))

X_w_noise = np.hstack([cancer.data, noise])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_w_noise, cancer.target, random_state=0, test_size=0.5
)


In [ ]:
select = SelectPercentile(percentile=50)
select.fit(X_train, y_train)

X_train_selected = select.transform(X_train)

print("X_train.shape:", X_train.shape)
print("X_train_selected.shape:", X_train_selected.shape)


- Feature count reduced by 50%
- We can inspect which features were kept using a Boolean mask


In [ ]:
import matplotlib.pyplot as plt

mask = select.get_support()
plt.matshow(mask.reshape(1, -1), cmap='gray_r')
plt.xlabel("Feature index")
plt.yticks(())


- Compare Logistic Regression:
  - Using all features
  - Using only selected features
- Removing noise features can improve accuracy


In [ ]:
from sklearn.linear_model import LogisticRegression

X_test_selected = select.transform(X_test)

lr = LogisticRegression(max_iter=10000)
lr.fit(X_train, y_train)
print("Score with all features:", lr.score(X_test, y_test))

lr.fit(X_train_selected, y_train)
print("Score with selected features:", lr.score(X_test_selected, y_test))


- Useful when:
  - Dataset has extremely many features
  - Many features are uninformative
- Results on real data are mixed
- Still valuable as a fast preprocessing step


### 4.7.2 Model-Based Feature Selection

- Uses a supervised model to estimate feature importance
- Keeps only the most important features
- Selection model does NOT have to match final model


- Tree-based models:
  - `feature_importances_`
- Linear models:
  - Absolute coefficient values
- L1-regularized linear models:
  - Produce sparse coefficients (implicit feature selection)


In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

select = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=42),
    threshold="median"
)


In [ ]:
select.fit(X_train, y_train)
X_train_model = select.transform(X_train)

print("X_train.shape:", X_train.shape)
print("X_train_model.shape:", X_train_model.shape)


In [ ]:
mask = select.get_support()
plt.matshow(mask.reshape(1, -1), cmap='gray_r')
plt.xlabel("Feature index")
plt.yticks(())


- Most original features were selected
- Some noise features remain
- More powerful than univariate selection


In [ ]:
X_test_model = select.transform(X_test)

score = LogisticRegression(max_iter=10000).fit(
    X_train_model, y_train
).score(X_test_model, y_test)

print("Test score:", score)


### 4.7.3 Iterative Feature Selection

- Builds multiple models with different feature subsets
- Two strategies:
  - Add features gradually
  - Remove features gradually
- Much more computationally expensive


- Starts with all features
- Trains a model
- Removes least important feature
- Repeats until desired number of features remains


In [ ]:
from sklearn.feature_selection import RFE

select = RFE(
    RandomForestClassifier(n_estimators=100, random_state=42),
    n_features_to_select=40
)

select.fit(X_train, y_train)


In [ ]:
mask = select.get_support()
plt.matshow(mask.reshape(1, -1), cmap='gray_r')
plt.xlabel("Feature index")
plt.yticks(())


- Better selection than previous methods
- Still misses one original feature
- Very slow:
  - Random forest trained many times


In [ ]:
X_train_rfe = select.transform(X_train)
X_test_rfe = select.transform(X_test)

score = LogisticRegression(max_iter=10000).fit(
    X_train_rfe, y_train
).score(X_test_rfe, y_test)

print("Test score:", score)


In [ ]:
print("Random forest score:", select.score(X_test, y_test))


- Automatic feature selection:
  - Simplifies models
  - Reduces overfitting
  - Improves interpretability
- Performance gains are usually modest
- Still an essential tool for feature engineering


# 4.8 Utilizing Expert Knowledge

Feature engineering is one of the most powerful places to apply expert (or common-sense) knowledge.
While machine learning models can learn patterns from data, they **cannot infer information that is
not explicitly encoded in the features**.

Adding expert-designed features:
- Does **not** force the model to use them
- Rarely hurts performance
- Often dramatically improves results, especially for simpler models

In this section, we use bike rental data to demonstrate how thoughtful feature design
can outperform naïve representations.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures


## Problem Setup: Citi Bike Rentals

The task is to predict how many bikes will be rented at a specific station
during each 3-hour interval.

The dataset contains:
- A timestamp (date + time)
- The number of bike rentals during the following 3 hours

We use the **first 23 days for training** and the **remaining 8 days for testing**.


In [ ]:
import mglearn

citibike = mglearn.datasets.load_citibike()
citibike.head()


## Visualization of the Time Series

The plot shows strong:
- Daily periodic patterns (day vs night)
- Weekly patterns (weekday vs weekend)

These patterns are obvious to humans — but **not automatically obvious to models**.


In [ ]:
plt.figure(figsize=(10, 3))
xticks = pd.date_range(start=citibike.index.min(),
                       end=citibike.index.max(), freq='D')
plt.xticks(xticks.astype("int"), xticks.strftime("%a %m-%d"),
           rotation=90, ha="left")
plt.plot(citibike, linewidth=1)
plt.xlabel("Date")
plt.ylabel("Rentals")
plt.show()


## Baseline Feature: POSIX Time (Bad Idea)

A common way to encode dates is POSIX time
(seconds since January 1, 1970).

However, **tree-based models cannot extrapolate** beyond the range of the training data.


In [ ]:
y = citibike.values
X = citibike.index.astype("int64").values.reshape(-1, 1) // 10**9

n_train = 184

def eval_on_features(features, target, regressor):
    X_train, X_test = features[:n_train], features[n_train:]
    y_train, y_test = target[:n_train], target[n_train:]

    regressor.fit(X_train, y_train)
    print("Test-set R^2: {:.2f}".format(regressor.score(X_test, y_test)))

    y_pred = regressor.predict(X_test)
    y_pred_train = regressor.predict(X_train)

    plt.figure(figsize=(10, 3))
    plt.plot(range(n_train), y_train, label="train")
    plt.plot(range(n_train, len(y_test) + n_train), y_test, label="test")
    plt.plot(range(n_train), y_pred_train, '--', label="prediction train")
    plt.plot(range(n_train, len(y_test) + n_train), y_pred, '--', label="prediction test")
    plt.legend()
    plt.show()


## Random Forest with POSIX Time

The model performs well on training data but completely fails on test data,
predicting a constant value.

This happens because all test timestamps are **outside the training range**.


In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=0)
eval_on_features(X, y, rf)


## Injecting Expert Knowledge: Time of Day

From inspection, bike rentals strongly depend on:
- Hour of the day
- Day of the week

We start by using **hour of the day** as a feature.


In [ ]:
X_hour = citibike.index.hour.values.reshape(-1, 1)
eval_on_features(X_hour, y, rf)


## Adding Day of the Week

Hourly patterns repeat differently on weekdays vs weekends.
Adding day-of-week significantly improves performance.


In [ ]:
X_hour_week = np.hstack([
    citibike.index.dayofweek.values.reshape(-1, 1),
    citibike.index.hour.values.reshape(-1, 1)
])

eval_on_features(X_hour_week, y, rf)


## Linear Models Fail with Integer Encoding

Using integers for day and hour treats them as continuous variables.
Linear models can only learn linear trends — which is incorrect here.


In [ ]:
eval_on_features(X_hour_week, y, LinearRegression())


## Correct Encoding: One-Hot Encoding

Day of week and hour of day are **categorical**, not continuous.
One-hot encoding allows the model to learn separate effects.


In [ ]:
enc = OneHotEncoder(sparse=False)
X_hour_week_oh = enc.fit_transform(X_hour_week)

eval_on_features(X_hour_week_oh, y, Ridge())


## Adding Interaction Features

To allow a separate effect for each (day, hour) combination,
we add interaction features.

This lets the model learn patterns like:
"Saturday at 6pm" ≠ "Monday at 6pm"


In [ ]:
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_interactions = poly.fit_transform(X_hour_week_oh)

eval_on_features(X_interactions, y, Ridge())


## Key Takeaways

- Models can only learn from the features you provide
- Dates should rarely be used as raw numbers
- Tree models cannot extrapolate beyond training ranges
- Linear models require correct feature encoding
- Expert knowledge can match complex models with simple, interpretable ones

**Good features + simple models > bad features + complex models**


# 4.9 Summary and Outlook

In this chapter, we explored how to work with different data types, with a particular focus on
**categorical variables**. We emphasized that data must be represented in a way that is compatible
with the machine learning algorithm being used—for example, by applying **one-hot encoding** to
categorical features.

We also discussed the importance of **feature engineering**, including the creation of new features
and the use of **expert or domain knowledge**. Carefully engineered features can often provide more
useful information than the original raw representation of the data.

In particular:
- **Linear models** can benefit greatly from feature transformations such as:
  - Binning
  - Polynomial features
  - Interaction terms
- **More complex, nonlinear models** (such as random forests and support vector machines) are often
  capable of learning complex relationships directly from the data, without explicitly expanding
  the feature space.

In practice, the **choice of features**—and how well they match the chosen learning algorithm—is
often the single most important factor in building an effective machine learning system.

Now that we have covered how to represent data appropriately and how to choose suitable algorithms,
the next chapter will focus on **evaluating model performance** and **selecting optimal parameter
settings**.

---

### Notes

1. This class underwent significant changes in version **0.20.0**, so be sure to use a recent version
   of *scikit-learn*.
2. This refers to the **Poisson distribution**, which is fundamental for modeling count data.
3. This is a crude approximation of **Poisson regression**, which would be the correct probabilistic
   approach.
